ready


In [13]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,precision_score
import glob
print("ready")
PREDICTORS = ["home_code", "away_code", "day_code", "hour"]
def preprocess(df):
    """"cleans and preprocess matches_data for the module"""
    #makes a copy to work on in order to not overload on the memory
    df = df.copy()
    #strips the dates
    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True,format="mixed")
    # making a code for every team
    df["home_code"] = df["HomeTeam"].astype("category").cat.codes
    df["away_code"] = df["AwayTeam"].astype("category").cat.codes
    # takes the date and hour cleanly
    df["day_code"] = df["Date"].dt.dayofweek
    if "Time" in df.columns:
        # strips the hour and turns it into an integer
        df["hour"] = df["Time"].str.replace(":.+", "", regex=True).fillna(15).astype(int)
    else:
        # default hour typical for English football
        df["hour"] = 15
    df["target"] = (df["FTR"] == "H").astype(int)
    return df

def make_prediction(data, predictors):
    #train and test partition
    train = data[data["Date"] < "2025-01-01"]
    test = data[data["Date"] >= "2025-01-01"]

    # module definition
    rf = RandomForestClassifier(n_estimators=50, min_samples_split=10, random_state=1)

    # testing and prediction
    rf.fit(train[predictors], train["target"])
    preds = rf.predict(test[predictors])

    # initialization of a table that shows the result and the truth
    combined = pd.DataFrame(dict(actual=test["target"], prediction=preds), index=test.index)

    return combined, precision_score(test["target"], preds)


#loading and concatenating the matches_data
file_paths = glob.glob("matches_data/*.csv")
all_seasons = [pd.read_csv(file) for file in file_paths]
df_combined = pd.concat(all_seasons, ignore_index=True)
#checking if the loading was successful
print(f"Loaded {len(file_paths)} files successfully!")
print(f"Total matches in dataset: {len(df_combined)}")

#processing the matches_data
df_preprocesed = preprocess(df_combined)

#predict and results
results, precision = make_prediction(df_preprocesed, PREDICTORS)
print(f"The module precision : {precision:.2%}")
print("\nThe results (Actual vs Prediction):")
print(results.head())






ready
Loaded 15 files successfully!
Total matches in dataset: 5612
The module precision : 48.70%

The results (Actual vs Prediction):
      actual  prediction
2468       0           0
2469       0           1
2470       1           0
2471       1           0
2472       0           0


/var/folders/jb/cqgbv6kn2cz13bs55k10gr380000gn/T/ipykernel_43080/2820475840.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["home_code"] = df["HomeTeam"].astype("category").cat.codes
/var/folders/jb/cqgbv6kn2cz13bs55k10gr380000gn/T/ipykernel_43080/2820475840.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["away_code"] = df["AwayTeam"].astype("category").cat.codes
/var/folders/jb/cqgbv6kn2cz13bs55k10gr380000gn/T/ipykernel_43080/2820475840.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually 